# Problem 1


Joan says she is itchy. There is a test for Allergy to Cats, but this test is not always right:

* For people that really do have the allergy, the test says "Yes" 80% of the time
* For people that do not have the allergy, the test says "Yes" 10% of the time

If 1% of the population has the allergy, and Joan's test says "Yes", what are the chances that Joan really has the allergy?

## Setup


In [1]:
import random
import pandas as pd
import numpy as np


In [2]:
# Parameters
n_simulations = 30_000_000      # number of simulations
p_allergy = 0.01                # 1% of population has the allergy
p_pos_given_allergy = 0.80      # 80% True Positive rate
p_pos_given_no_allergy = 0.10   # 10% False Positive rate


## Monte Carlo simulation with Python



In [3]:
# Store the results in an array (list)
# Each entry will be a dictionary: {"has_allergy": bool, "test_result": bool}
results = []

# Set a seed for consistency
random.seed(42)

for _ in range(n_simulations):
    # Step 1: Does this person have the allergy?
    has_allergy = random.random() < p_allergy

    # Step 2: What is the test result?
    if has_allergy:
        # If they have the allergy, 80% test positive
        test_positive = random.random() < p_pos_given_allergy
    else:
        # If they don't have the allergy, 10% test positive (false positive)
        test_positive = random.random() < p_pos_given_no_allergy

    # Step 3: Store the result in our array
    results.append({
        "has_allergy": has_allergy,
        "test_positive": test_positive
    })


In [4]:
results[:10]

[{'has_allergy': False, 'test_positive': True},
 {'has_allergy': False, 'test_positive': False},
 {'has_allergy': False, 'test_positive': False},
 {'has_allergy': False, 'test_positive': True},
 {'has_allergy': False, 'test_positive': True},
 {'has_allergy': False, 'test_positive': False},
 {'has_allergy': False, 'test_positive': False},
 {'has_allergy': False, 'test_positive': False},
 {'has_allergy': False, 'test_positive': False},
 {'has_allergy': False, 'test_positive': True}]

In [5]:
# convert to a data frame
df = pd.DataFrame(results)
df.shape

(30000000, 2)

In [6]:
df


,has_allergy,test_positive
0,False,True
1,False,False
2,False,False
3,False,True
4,False,True
...,...,...
29999995,False,False
29999996,False,True
29999997,False,False
29999998,False,False


In [7]:
vc = df.value_counts()
vc.to_frame(name='count').style.format('{:,}')


#### Bayes



##### Calculate conditional and marginal probabilities.

In [8]:
p_has_allergy = ( vc[True,:].sum() ) / vc.sum()
p_has_allergy

np.float64(0.009973466666666667)

In [9]:
p_test_positive = ( vc[:,True].sum() ) / vc.sum()
p_test_positive

np.float64(0.10699286666666667)

In [10]:
p_test_pos_given_has_allergy = vc[True,True] / ( vc[True,:].sum() )
p_test_pos_given_has_allergy


np.float64(0.7999725939492788)

##### Assemble using Bayes theorem.

In [11]:
p_has_allergy_given_test_post = p_test_pos_given_has_allergy * p_has_allergy / p_test_positive
p_has_allergy_given_test_post


np.float64(0.074570391920209)

#### Calculate directly using counts.

In [12]:
vc[True, True]/ (vc[:, True].sum())

np.float64(0.074570391920209)

## Redo using numpy


In [13]:
# Set a seed for consistency
np.random.seed(42)

pop = np.random.rand(n_simulations)
pop[:100]

array([0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864,
       0.15599452, 0.05808361, 0.86617615, 0.60111501, 0.70807258,
       0.02058449, 0.96990985, 0.83244264, 0.21233911, 0.18182497,
       0.18340451, 0.30424224, 0.52475643, 0.43194502, 0.29122914,
       0.61185289, 0.13949386, 0.29214465, 0.36636184, 0.45606998,
       0.78517596, 0.19967378, 0.51423444, 0.59241457, 0.04645041,
       0.60754485, 0.17052412, 0.06505159, 0.94888554, 0.96563203,
       0.80839735, 0.30461377, 0.09767211, 0.68423303, 0.44015249,
       0.12203823, 0.49517691, 0.03438852, 0.9093204 , 0.25877998,
       0.66252228, 0.31171108, 0.52006802, 0.54671028, 0.18485446,
       0.96958463, 0.77513282, 0.93949894, 0.89482735, 0.59789998,
       0.92187424, 0.0884925 , 0.19598286, 0.04522729, 0.32533033,
       0.38867729, 0.27134903, 0.82873751, 0.35675333, 0.28093451,
       0.54269608, 0.14092422, 0.80219698, 0.07455064, 0.98688694,
       0.77224477, 0.19871568, 0.00552212, 0.81546143, 0.70685

In [14]:
# create population with allergy
has_allergy = pop < p_allergy
has_allergy[:100]


array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False])

In [15]:
# Assign the correct test probability for each person based on whether they have the allergy
test_probs = np.where(has_allergy, p_pos_given_allergy, p_pos_given_no_allergy)
test_probs[:100]


array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
       0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
       0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
       0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
       0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
       0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.8, 0.1, 0.1, 0.1, 0.1, 0.1,
       0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,
       0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

In [16]:
# Step 3: Generate all test results at once by comparing random draws against those probabilities
test_positive = np.random.rand(n_simulations) < test_probs
test_positive[:100]


array([False,  True, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False,  True, False, False, False, False, False, False, False,
       False])

In [17]:
data_dict = {
    "population"    : pop,
    "has_allergy"   : has_allergy,
    "test_probs"    : test_probs,
    "test_positive" : test_positive,
}

In [18]:
data_dict


{'population': array([0.37454012, 0.95071431, 0.73199394, ..., 0.28813501, 0.31319477,
        0.15101154]),
 'has_allergy': array([False, False, False, ..., False, False, False]),
 'test_probs': array([0.1, 0.1, 0.1, ..., 0.1, 0.1, 0.1]),
 'test_positive': array([False,  True, False, ..., False, False, False])}

In [19]:
df = pd.DataFrame( data_dict )
df


,population,has_allergy,test_probs,test_positive
0,0.374540,False,0.1,False
1,0.950714,False,0.1,True
2,0.731994,False,0.1,False
3,0.598658,False,0.1,False
4,0.156019,False,0.1,False
...,...,...,...,...
29999995,0.156686,False,0.1,False
29999996,0.640828,False,0.1,False
29999997,0.288135,False,0.1,False
29999998,0.313195,False,0.1,False


In [20]:
vc = df[["has_allergy","test_positive"]].value_counts()
vc.to_frame(name='count').style.format('{:,}')


#### Bayes



##### Calculate conditional and marginal probabilities.


In [21]:
p_has_allergy = ( vc[True,:].sum() ) / vc.sum()
p_has_allergy

np.float64(0.009981233333333334)

In [22]:
p_test_positive = ( vc[:,True].sum() ) / vc.sum()
p_test_positive

np.float64(0.10701856666666666)

In [23]:
p_test_pos_given_has_allergy = vc[True,True] / ( vc[True,:].sum() )
p_test_pos_given_has_allergy


np.float64(0.8000146942428624)

##### Assemble using Bayes theorem.

In [24]:
p_has_allergy_given_test_post = p_test_pos_given_has_allergy * p_has_allergy / p_test_positive
p_has_allergy_given_test_post


np.float64(0.07461446720927241)

#### Calculate directly using counts.

In [25]:
vc[True, True]/ (vc[:, True].sum())

np.float64(0.07461446720927241)

## Redo using Pandas


In [26]:
df=pd.DataFrame()
df


""


In [27]:
# Set a seed for consistency
np.random.seed(42)

df["population"] =  np.random.rand(n_simulations)
df


,population
0,0.374540
1,0.950714
2,0.731994
3,0.598658
4,0.156019
...,...
29999995,0.156686
29999996,0.640828
29999997,0.288135
29999998,0.313195


In [28]:
df["allergy_probs"] = p_allergy
df


,population,allergy_probs
0,0.374540,0.01
1,0.950714,0.01
2,0.731994,0.01
3,0.598658,0.01
4,0.156019,0.01
...,...,...
29999995,0.156686,0.01
29999996,0.640828,0.01
29999997,0.288135,0.01
29999998,0.313195,0.01


In [29]:
# create population with allergy
df["has_allergy"] = df["population"] < df["allergy_probs"]
df


,population,allergy_probs,has_allergy
0,0.374540,0.01,False
1,0.950714,0.01,False
2,0.731994,0.01,False
3,0.598658,0.01,False
4,0.156019,0.01,False
...,...,...,...
29999995,0.156686,0.01,False
29999996,0.640828,0.01,False
29999997,0.288135,0.01,False
29999998,0.313195,0.01,False


In [30]:
(
df[["allergy_probs", "has_allergy"]]
  .value_counts()
  .to_frame(name='count')
  .style.format('{:,}')
)

In [31]:
# Assign the correct test probability for each person based on whether they have the allergy

df["test_probs"] = np.where(df["has_allergy"], p_pos_given_allergy, p_pos_given_no_allergy)
df


,population,allergy_probs,has_allergy,test_probs
0,0.374540,0.01,False,0.1
1,0.950714,0.01,False,0.1
2,0.731994,0.01,False,0.1
3,0.598658,0.01,False,0.1
4,0.156019,0.01,False,0.1
...,...,...,...,...
29999995,0.156686,0.01,False,0.1
29999996,0.640828,0.01,False,0.1
29999997,0.288135,0.01,False,0.1
29999998,0.313195,0.01,False,0.1


In [32]:
df["pop2"] =  np.random.rand(df.index.size)
df


,population,allergy_probs,has_allergy,test_probs,pop2
0,0.374540,0.01,False,0.1,0.773222
1,0.950714,0.01,False,0.1,0.022870
2,0.731994,0.01,False,0.1,0.135256
3,0.598658,0.01,False,0.1,0.547153
4,0.156019,0.01,False,0.1,0.112734
...,...,...,...,...,...
29999995,0.156686,0.01,False,0.1,0.811033
29999996,0.640828,0.01,False,0.1,0.564256
29999997,0.288135,0.01,False,0.1,0.819708
29999998,0.313195,0.01,False,0.1,0.314705


In [33]:
# Step 3: Generate all test results at once by comparing random draws against those probabilities
df["test_positive"] = df["pop2"] < df["test_probs"]
df


,population,allergy_probs,has_allergy,test_probs,pop2,test_positive
0,0.374540,0.01,False,0.1,0.773222,False
1,0.950714,0.01,False,0.1,0.022870,True
2,0.731994,0.01,False,0.1,0.135256,False
3,0.598658,0.01,False,0.1,0.547153,False
4,0.156019,0.01,False,0.1,0.112734,False
...,...,...,...,...,...,...
29999995,0.156686,0.01,False,0.1,0.811033,False
29999996,0.640828,0.01,False,0.1,0.564256,False
29999997,0.288135,0.01,False,0.1,0.819708,False
29999998,0.313195,0.01,False,0.1,0.314705,False


In [34]:
vc = df[["has_allergy","test_positive"]].value_counts()
vc.to_frame(name='count').style.format('{:,}')


#### Bayes



##### Calculate conditional and marginal probabilities.


In [35]:
p_has_allergy = ( vc[True,:].sum() ) / vc.sum()
p_has_allergy

np.float64(0.009981233333333334)

In [36]:
p_test_positive = ( vc[:,True].sum() ) / vc.sum()
p_test_positive

np.float64(0.10701856666666666)

In [37]:
p_test_pos_given_has_allergy = vc[True,True] / ( vc[True,:].sum() )
p_test_pos_given_has_allergy


np.float64(0.8000146942428624)

##### Assemble using Bayes theorem.

In [38]:
p_has_allergy_given_test_post = p_test_pos_given_has_allergy * p_has_allergy / p_test_positive
p_has_allergy_given_test_post


np.float64(0.07461446720927241)

#### Calculate directly using counts.

In [39]:
vc[True, True]/ (vc[:, True].sum())

np.float64(0.07461446720927241)